# 🧰 스킬 확장 & MCP (Skills & Model Context Protocol)

본 실습 노트북은 에이전트의 도구 하네스의 확장 방법인 **Skill**과 **Model Context Protocol (MCP)** 의 2대 바인딩 아키텍처(**Static Process Binding** vs **Dynamic Context Binding**)를 단계별로 직접 실행하며 학습하는 자료입니다.

---

### 💡 핵심 아이디어

**도구 과부하(Tool Overload) 극복과 Progressive Disclosure (점진적 도구 노출)**
   - 수십~수백 개의 전문 도구를 LLM 시스템 프롬프트에 무차별 주입하면 **도구 오호출(Tool Hallucination)의 증가와 토큰 비용/지연 시간이 폭증**합니다.
   - 따라서 최소한의 기본 도구(`file_read`, `bash_command` 등)만 장착하고, 필요한 순간에만 `skills/` 디렉터리의 문서를 열람하여 스크립트를 자율 실행하는 **점진적 도구 노출**이 활발하게 활용되고 있습니다.
   - 에이전트에게 도구는 행동(Action)을 의미합니다. **도구가 이전처럼 에이전트의 프로세스에 포함되는 것이 아닌 컨텍스트로서 존재**하면 도구는 에이전트의 읽기 작업을 통해 효율적으로 확장될 수 있습니다. 읽은 도구는 실행(bash_command)로 실행할 수 있습니다.


---

### 🎓 학습 목차 (Curriculum Flow)

| 파트 | 주제 | 핵심 실습 내용 |
|:---:|:---|:---|
| **Step 0** | **환경 세팅 & 격리 샌드박스** | 루트 탐색, `.env` 로드, `nest_asyncio`, 실습 샌드박스(`demo_dir`, 가상 PDF/MCP) 생성 |
| **Part 1** | **Skills & Progressive Disclosure** | 1-1) `SkillPromptBuilder` YAML Frontmatter 고속 스캔 실습<br>1-2) Base Tools (`file_read` + `bash_command`) 기반 PDF 스킬 자율 발견 & 실행 |
| **Part 2** | **MCP 개념 & 2대 바인딩 아키텍처** | 2-1) Model Context Protocol 표준 통신 규격 (Client ↔ Server)<br>2-2) Static Process Binding vs Dynamic Context Binding 특성 비교 |
| **Part 3** | **[패턴 1] Static Process Binding 실습** | 3-1) FastMCP 클라이언트 기반 `list_mcp_tools`, `execute_mcp_tool` 도구 정의<br>3-2) 전용 도구를 장착한 에이전트의 MCP 서버 도구 발견 & 실행 궤적 확인 |
| **Part 4** | **[패턴 2] Dynamic Context Binding (Progressive MCP)** | 4-1) 전용 도구 없는 초경량 에이전트에 `skills/mcp/` 가이드 주입<br>4-2) `list_tools.py` & `execute_tool.py` 스크립트를 자율 구동하여 100% 동일 미션 완수<br>4-3) 도구 수 폭증 시 토큰 효율성 비교 |
| **Part 5** | **[학습 정리] 프로덕션 도구 확장 3대 황금률** | 네이티브 Tool vs Skills vs MCP 선택 기준 및 의사결정 매트릭스 |
| **Part 6** | **🧹 Clean-up & Reset** | 샌드박스 임시 디렉토리 및 프로세스 완전 정리 |

---


## 🛠️ Step 0. 환경 세팅 & 격리 샌드박스(Sandbox) 초기화

주피터 환경에서 비동기 루프를 안전하게 실행하기 위해 `nest_asyncio`를 활성화하고, **실습 전용 독립 임시 디렉토리(`demo_dir`)**를 생성합니다.

실습에 필요한 **가상 PDF 파일**과 **모의 FastMCP 로컬 서버 스크립트**를 샌드박스 내에 동적으로 생성합니다.

In [ ]:
import os
import sys
import json
import shutil
import asyncio
import tempfile
import nest_asyncio
from dotenv import load_dotenv

# 1. 비동기 루프 패치 (Jupyter 환경 필수)
nest_asyncio.apply()

# 2. 프로젝트 루트 상향 동적 탐색 (어느 서브 폴더에서 실행해도 100% 작동)
def find_project_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, "app")) and (
            os.path.exists(os.path.join(p, ".env")) or os.path.exists(os.path.join(p, "configs"))
        ):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.join(os.getcwd(), ".."))

project_root = find_project_root()
os.chdir(project_root)
print(f"🔄 작업 디렉토리를 프로젝트 루트('{os.getcwd()}')로 전환 완료.\n")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 3. 환경 변수 로드
dotenv_path = os.path.join(project_root, ".env")
load_dotenv(dotenv_path, override=True)

# 4. 실습 격리용 샌드박스 디렉토리 생성
demo_dir = tempfile.mkdtemp(prefix="agent_skills_mcp_lab_")
demo_data_dir = os.path.join(demo_dir, "data")
demo_server_dir = os.path.join(demo_dir, "server")
os.makedirs(demo_data_dir, exist_ok=True)
os.makedirs(demo_server_dir, exist_ok=True)

# 5. 실습용 실제 바이너리 PDF 논문 파일 생성 (PyMuPDF / pypdf)
demo_pdf_path = os.path.join(demo_data_dir, "research_paper.pdf")
pdf_created = False

try:
    try:
        import pymupdf as fitz
    except ImportError:
        import fitz

    doc = fitz.open()
    # 1페이지 (Abstract & Introduction)
    p1 = doc.new_page()
    p1_text = """[Research Paper] Frontier Agent Harness & Progressive Skill Execution\nAuthor: Dr. Kim & AI Engineering Team\nPages: 2\nSubject: Agentic AI Systems and Context Management\n\n1. Abstract\nThis paper proposes a 5-layer prompt stack and progressive skill disclosure pattern for autonomous AI agents. By maintaining minimal base tools (file_read and bash_command), the agent achieves 99% token efficiency and eliminates tool-overload hallucinations.\n\n2. Introduction & Problem Definition\nTraditional agents inject 50+ tool definitions directly into system prompts, resulting in attention drift and extreme token consumption. In contrast, our harness dynamically discovers tools through lightweight YAML Frontmatter metadata."""
    p1.insert_text((50, 72), p1_text, fontsize=11)

    # 2페이지 (Evaluation & Conclusion)
    p2 = doc.new_page()
    p2_text = """3. Experimental Evaluation\nWe evaluated the progressive skill pattern across 500 complex analytical tasks. The progressive skill disclosure pattern achieved a 100% task completion rate with an average latency reduction of 73% compared to monolithic tool injection.\n\n4. Conclusion & Future Work\nBy decoupling domain scripts from model prompts and leveraging standardized MCP servers, production agents can scale indefinitely without context saturation."""
    p2.insert_text((50, 72), p2_text, fontsize=11)

    doc.set_metadata({
        "title": "Frontier Agent Harness & Progressive Skill Execution",
        "author": "Dr. Kim & AI Engineering Team",
        "subject": "Agentic AI Systems and Context Management"
    })
    doc.save(demo_pdf_path)
    doc.close()
    pdf_created = True
    print("📄 실제 바이너리 PDF 파일 생성 완료 (2 Pages, PyMuPDF)")
except Exception as e:
    pass

if not pdf_created:
    with open(demo_pdf_path, "w", encoding="utf-8") as f:
        f.write("""[Virtual PDF Container]\ntitle: Frontier Agent Harness & Progressive Skill Execution\nauthor: Dr. Kim & AI Engineering Team\npages: 2\nsubject: Agentic AI Systems and Context Management\ncontent: This paper proposes a 5-layer prompt stack and progressive skill disclosure pattern. By maintaining small base tools (file_read and bash), the agent achieves 99% token efficiency and zero tool-overload hallucination. Evaluation results show 100% task completion rate on complex web scraping and analytical workloads.\n""")
    print("📄 가상 PDF 컨테이너 파일 생성 완료")

# 6. 실습용 로컬 FastMCP Stdio 서버 스크립트 생성 (Data & Math MCP Server)
demo_mcp_server_path = os.path.join(demo_server_dir, "demo_mcp_server.py")
with open(demo_mcp_server_path, "w", encoding="utf-8") as f:
    f.write("""#!/usr/bin/env python
import sys
from fastmcp import FastMCP

mcp = FastMCP("Data & Math Engine")

@mcp.tool()
def calculate_roi(revenue: float, cost: float) -> str:
    \"\"\"Calculates Return on Investment (ROI) percentage given revenue and cost.\"\"\"
    if cost == 0:
        return "Error: Cost cannot be zero."
    roi = ((revenue - cost) / cost) * 100.0
    return f"Calculated ROI: {roi:.2f}% (Revenue: ${revenue:,.2f}, Cost: ${cost:,.2f})"

@mcp.tool()
def query_database(table: str, limit: int = 5) -> str:
    \"\"\"Queries records from a simulated corporate database table (users, sales, products).\"\"\"
    sample_data = {
        "sales": [
            {"quarter": "Q1", "revenue": 1200000, "target_met": True},
            {"quarter": "Q2", "revenue": 1450000, "target_met": True},
            {"quarter": "Q3", "revenue": 1100000, "target_met": False},
            {"quarter": "Q4", "revenue": 1800000, "target_met": True}
        ],
        "users": [
            {"id": 1, "name": "Cheolsu Kim", "role": "Lead Engineer"},
            {"id": 2, "name": "Younghee Lee", "role": "Data Scientist"}
        ]
    }
    records = sample_data.get(table.lower(), [{"info": f"Table '{table}' is empty or not found."}])
    return str(records[:limit])

if __name__ == '__main__':
    mcp.run(transport='stdio')
""")

# 7. 프로덕션 도구 및 LLM 팩토리 임포트
from app.tools.common import file_read, bash_command, file_writer
from app.utils import init_chat_model
from app.middleware.prompt import SkillPromptBuilder
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# 8. 실시간 실행 궤적 시각화 디버거 헬퍼 함수
def stream_and_debug_agent(agent, inputs, config=None, agent_name="Agent"):
    print(f"\n{'='*70}")
    print(f"🤖 [{agent_name}] 실시간 추적 시작")
    print(f"{'='*70}")
    
    final_response = ""
    for chunk in agent.stream(inputs, config=config, stream_mode="values"):
        messages = chunk.get("messages", [])
        if not messages:
            continue
        last_msg = messages[-1]
        
        # Tool Call 감지
        if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
            for tc in last_msg.tool_calls:
                print(f"\n👉 [Tool Call] {tc['name']}")
                print(f"   Args: {json.dumps(tc['args'], ensure_ascii=False)}")
        # Tool Execution 결과
        elif isinstance(last_msg, ToolMessage):
            preview = str(last_msg.content).strip()
            if len(preview) > 300:
                preview = preview[:300] + " ... [TRUNCATED]"
            print(f"\n   ↳ [Tool Result] {preview}")
        # 최종 LLM 응답
        elif isinstance(last_msg, AIMessage) and not getattr(last_msg, "tool_calls", None):
            final_response = last_msg.content

    print(f"\n{'='*70}")
    print(f"💬 [{agent_name}] 최종 응답:\n{final_response}")
    print(f"{'='*70}\n")
    return final_response

print(f"✅ 환경 설정 및 격리 샌드박스 초기화 완료!")
print(f"  - Project Root       : {project_root}")
print(f"  - Demo Sandbox      : {demo_dir}")
print(f"  - Actual PDF Path   : {demo_pdf_path}")
print(f"  - Demo FastMCP Server: {demo_mcp_server_path}")


## 📂 Part 1. Skills & Progressive Disclosure (점진적 도구 노출)

### 1. 왜 Progressive Disclosure인가?
- 에이전트에게 50개 이상의 도구를 한꺼번에 주면, 모델은 어떤 도구를 써야 할지 혼란을 겪고 **엉뚱한 도구를 호출하거나 인자 형식을 왜곡**합니다.
- **Progressive Disclosure 원칙**:
  1. 에이전트에게는 **가장 기본이 되는 연동 도구(Base Tools: `file_read`, `bash_command`)**만 상시 제공합니다.
  2. `skills/` 디렉터리에 각 전문 작업별 가이드 문서(`SKILL.md`)와 파이썬 실행 스크립트(`scripts/`)를 모듈화하여 저장합니다.
  3. 에이전트는 필요할 때 스스로 `SKILL.md`를 열람하여 파라미터를 확인하고, `bash_command`로 스크립트를 가동합니다.

- **Frontmatter 스캐너와 초경량 스킬 카탈로그 (`SkillPromptBuilder`)**

    다만 점진적 노출이더라도 에이전트가 최소한 어떤 스킬이 가능한지 알아야 합니다. 이를 효율적으로 처리하기 위해 각 스킬 폴더에는 SKILL.md 파일이 있고, 해당 문서의 상단에 Frontmatter라고 하는 `미리보기`에 해당하는 부분이 있습니다.

    각 스킬 폴더의 `SKILL.md` 상단 **YAML Frontmatter**(`name`, `description`)만 2KB I/O로 고속 스캔하여 프롬프트 상단(Layer 2)에 수십 자 내외의 경량 카탈로그로 등록합니다.


In [ ]:
# Part 1-1. SkillPromptBuilder의 YAML Frontmatter 스캔 및 카탈로그 자동 생성 실습

# 1. SkillPromptBuilder 인스턴스 생성 (skills/ 디렉터리 스캔)
skill_builder = SkillPromptBuilder(
    skills_dirs=[os.path.join(project_root, "skills")],
    guidelines_path=os.path.join(project_root, "app/prompts/SKILL.md") if os.path.exists(os.path.join(project_root, "app/prompts/SKILL.md")) else None
)

In [ ]:
# 2. 스캔된 <skills> 카탈로그 텍스트 확인
catalog_text = skill_builder.build_catalog()
print("📦 [스캔된 Skills 카탈로그]")
print(catalog_text)
print("\n" + "-"*50 + "\n")

In [ ]:
# 3. 가이드라인과 결합된 최종 주입 문자열 확인
full_skill_prompt = skill_builder.assemble()
print("📋 [최종 스킬 조립 프롬프트 미리보기 (상위 500자)]")
print(full_skill_prompt)

### 2. Base Tools를 장착한 에이전트의 PDF 스킬 자율 발견 & 실행

에이전트에게 **오직 `[file_read, bash_command]` 2대 도구만** 부여하고, `skills/pdf_processing/Skill.md`를 스스로 찾아 읽어 가상 PDF 문서의 텍스트를 추출하도록 지시합니다.

In [ ]:
# Part 1-2. Base Tools 기반 Progressive Skill 실행 실습

# 1. LLM 초기화 (gemini-3.7-flash)
llm = init_chat_model(model="gemini-3.7-flash", temperature=0.0)

# 2. 에이전트에게 주어지는 기본 도구 (단 2종!)
base_tools = [file_read, bash_command]

# 3. Progressive Skill 에이전트 시스템 지침 정의
system_instruction = f"""You are an expert AI agent with progressive skill execution capabilities.
You ONLY have 'file_read' and 'bash_command' base tools.

{full_skill_prompt}

When a user asks you to handle a specific domain task (e.g. PDF parsing, MCP, analytics):
1. Check the Available Skills Catalog to find the relevant skill path (e.g. 'skills/pdf_processing/Skill.md').
2. Read the skill's documentation using 'file_read' to learn the exact script usage and parameters.
3. Run the chosen python script using 'bash_command' to obtain the result.
4. Present the extracted information clearly to the user.
"""

# 4. 에이전트 생성
checkpointer = MemorySaver()
progressive_agent = create_agent(
    model=llm,
    tools=base_tools,
    system_prompt=system_instruction,
    checkpointer=checkpointer
)

# 5. PDF 텍스트 추출 질의 실행
relative_pdf_path = os.path.relpath(demo_pdf_path, project_root).replace("\\", "/")
user_query = f"'{relative_pdf_path}' 논문 파일의 제목, 저자, 페이지 수 및 주요 요약 내용을 추출해서 한글로 정리해줘."

inputs = {"messages": [HumanMessage(content=user_query)]}
config = {"configurable": {"thread_id": "skill_demo_session_1"}}

stream_and_debug_agent(progressive_agent, inputs, config=config, agent_name="Progressive Skill Agent")

## 🔌 Part 2. Model Context Protocol (MCP) 개요 & 2대 바인딩 모델

**Model Context Protocol (MCP)** 은 Anthropic에서 오픈소스로 발표한 **AI 에이전트와 외부 도구/데이터 소스 간의 양방향 표준 통신 규약**입니다.

### 1. Static Process Binding vs Dynamic Context Binding 비교

| 비교 항목 | ⚙️ 패턴 1: Static Process Binding (정적 바인딩) | ⚡ 패턴 2: Dynamic Context Binding (동적 바인딩) |
|:---|:---|:---|
| **개념** | MCP 서버의 도구를 에이전트 **프로세스 툴킷에 직접 등록** | 도구를 프로세스가 아닌 **프롬프트/파일시스템 컨텍스트**로 다룸 |
| **구현 방식** | `list_mcp_tools`, `execute_mcp_tool` 전용 도구 바인딩 | 기본 도구(`file_read`, `bash`) + `skills/mcp/` 실행 스크립트 |
| **장점** | 도구 수가 적을 때 빠르고 직관적인 LLM 호출 가능 | 도구가 수백 개로 늘어나도 **프롬프트 오버로드가 전혀 발생하지 않음** |
| **단점** | 도구 수가 많아지면 스키마 비용 및 레이턴시 증가 | 2단계(문서 열람 ➔ 스크립트 실행) 추론 오버헤드 1회 발생 |
| **권장 사용처** | 고빈도 핵심 도구 (자주 쓰는 3~5개 MCP 기능) | 외부 대규모 도구 생태계 연동 (Wikipedia, DB, Cloud 등) |

## ⚙️ Part 3. [패턴 1] Static Process Binding 실습

`fastmcp.Client`를 활용하여 원격 SSE 또는 로컬 Stdio MCP 서버와 실시간 통신하는 2종의 전용 도구(`list_mcp_tools`, `execute_mcp_tool`)를 정의하고 에이전트에 직접 장착합니다.

In [ ]:
# Part 3-1. FastMCP 기반 전용 MCP 도구 2종 정의 (Hybrid SSE / Stdio)
import shlex
from fastmcp import Client
from fastmcp.client.transports import StdioTransport
from langchain_core.tools import tool

def get_mcp_client(target: str) -> Client:
    """URL이면 Remote SSE Client, 커맨드라인이면 Local Stdio Client를 반환합니다."""
    if target.startswith("http://") or target.startswith("https://"):
        return Client(target)
    else:
        parts = shlex.split(target)
        transport = StdioTransport(command=parts[0], args=parts[1:])
        return Client(transport)

@tool
def list_mcp_tools(target: str) -> str:
    """지정된 MCP 서버(HTTP SSE URL 또는 Stdio 구동 명령어)의 모든 도구 목록과 파라미터 스키마를 실시간 조회합니다.
    
    Args:
        target: MCP 서버의 SSE URL (예: 'https://...') 또는 Stdio 실행 명령어 (예: 'python server.py')
    """
    async def _run():
        client = get_mcp_client(target)
        async with client:
            tools_response = await client.list_tools()
            tools_list = []
            for t in tools_response:
                tools_list.append({
                    "name": t.name,
                    "description": t.description,
                    "input_schema": t.inputSchema
                })
            return json.dumps({"status": "SUCCESS", "tools": tools_list}, ensure_ascii=False, indent=2)
            
    try:
        return asyncio.run(_run())
    except Exception as e:
        return f"Error retrieving MCP tools from '{target}': {str(e)}"

@tool
def execute_mcp_tool(target: str, tool_name: str, arguments: dict) -> str:
    """지정된 MCP 서버의 특정 도구를 입력 파라미터(arguments)로 실행하고 결과를 반환합니다.
    
    Args:
        target: MCP 서버의 SSE URL 또는 Stdio 실행 명령어
        tool_name: 실행할 MCP 도구 이름
        arguments: 도구 실행에 필요한 JSON 인자 딕셔너리
    """
    async def _run():
        client = get_mcp_client(target)
        async with client:
            result = await client.call_tool(tool_name, arguments)
            content_str = str(result.content)
            if hasattr(result, "content") and isinstance(result.content, list):
                content_str = "\n".join([item.text for item in result.content if hasattr(item, "text")])
            return content_str
            
    try:
        return asyncio.run(_run())
    except Exception as e:
        return f"Error executing MCP tool '{tool_name}' on '{target}': {str(e)}"

print("✅ list_mcp_tools, execute_mcp_tool 도구 정의 완료!")

In [ ]:
# Part 3-2. Static Process Binding 에이전트 구축 및 로컬 FastMCP 서버 연동 실행

# 1. MCP 전용 도구 바인딩
mcp_direct_tools = [list_mcp_tools, execute_mcp_tool]

# 2. 상대 경로 명령어 생성
rel_server_cmd = f"python {os.path.relpath(demo_mcp_server_path, project_root).replace(chr(92), '/')}"

static_mcp_prompt = f"""You are an assistant equipped with direct MCP interface tools.
Available active MCP server command:
- Data & Math Engine: '{rel_server_cmd}'

To solve the user's request:
1. Use 'list_mcp_tools' to discover the available tools and input schemas on the server.
2. Use 'execute_mcp_tool' to invoke the required tools with proper arguments.
3. Present the calculated/retrieved results clearly to the user in Korean.
"""

static_mcp_agent = create_agent(
    model=llm,
    tools=mcp_direct_tools,
    system_prompt=static_mcp_prompt,
    checkpointer=MemorySaver()
)

# 3. 기업 매출 데이터 조회 및 ROI 계산 복합 미션 수행
user_query_mcp = (
    "로컬 FastMCP 서버에서 sales 테이블의 매출 데이터를 조회한 뒤, "
    "Q4 분기 매출(Revenue)과 비용(Cost=$1,200,000)을 기준으로 ROI를 계산해서 최종 보고해줘."
)

inputs = {"messages": [HumanMessage(content=user_query_mcp)]}
config = {"configurable": {"thread_id": "static_mcp_session"}}

stream_and_debug_agent(static_mcp_agent, inputs, config=config, agent_name="Static MCP Binding Agent")

## ⚡ Part 4. [패턴 2] Dynamic Context Binding (Progressive MCP) 실습

이번에는 에이전트에게 **MCP 전용 도구를 일체 주지 않고**, 오직 **기본 도구(`file_read`, `bash_command`)**와 `skills/mcp/Skill.md` 가이드라인만을 제공합니다.

에이전트가 `skills/mcp/scripts/list_tools.py`와 `execute_tool.py`를 스스로 발견하고 호출하여 **완전한 기능적 동등성(Parity)**을 달성하는 과정을 관찰합니다.

In [ ]:
# Part 4. Dynamic Context Binding (Progressive MCP) 실습

# 1. MCP 컨텍스트 카탈로그 정의 (프롬프트 주입용)
mcp_context_catalog = f"""====== ACTIVE MCP SERVERS CATALOG ======
- Server Name: Data & Math Engine
- Launch Command: {rel_server_cmd}
- Description: Provides database query and financial math calculations.
========================================"""

# 2. Progressive MCP 시스템 지침 구성
dynamic_mcp_prompt = f"""You are a progressive skill-disclosure agent.
You only have 'file_read' and 'bash_command' base tools.

{mcp_context_catalog}

Guidelines for MCP Interaction:
1. When you need to interact with an MCP server listed in the CATALOG, first read 'skills/mcp/Skill.md' using 'file_read'.
2. Discover available tools by running 'python skills/mcp/scripts/list_tools.py --url "<TARGET_CMD_OR_URL>"' with 'bash_command'.
3. Execute tools by running 'python skills/mcp/scripts/execute_tool.py --url "<TARGET>" --tool "<NAME>" --args "<JSON>"' with 'bash_command'.
4. Return the final structured summary to the user.
"""

# 3. 오직 Base Tools만 장착한 에이전트 생성
dynamic_mcp_agent = create_agent(
    model=llm,
    tools=[file_read, bash_command],
    system_prompt=dynamic_mcp_prompt,
    checkpointer=MemorySaver()
)

# 4. 동일한 복합 미션 수행 (sales 조회 + ROI 계산)
user_query_dynamic = (
    "Data & Math Engine MCP 서버에서 sales 테이블의 데이터를 조회하고, "
    "Q4 분기 매출과 비용(Cost=$1,200,000)에 대한 ROI를 계산해서 한글로 보고해줘."
)

inputs = {"messages": [HumanMessage(content=user_query_dynamic)]}
config = {"configurable": {"thread_id": "dynamic_mcp_session"}}

stream_and_debug_agent(dynamic_mcp_agent, inputs, config=config, agent_name="Dynamic Context MCP Agent")

## 📝 Part 5. [학습 정리] 프로덕션 도구 확장 3대 황금률

실무에서 수십 개의 API와 도구를 에이전트에 통합할 때 따르는 **3대 아키텍처 원칙**입니다:

```
┌─────────────────────────────────────────────────────────────────────────────┐
│ 1. 범용 도구 (8~10개)       ➔ 네이티브 LangChain Tool 바인딩             │
│    • file_read, file_writer, bash_command, glob_search, grep_search 등        │
├─────────────────────────────────────────────────────────────────────────────┤
│ 2. 도메인 특화 비즈니스 로직   ➔ Skills (SKILL.md + Standalone Scripts)    │
│    • PDF 파싱, 데이터 프로파일링, Excel 생성 등                             │
│    • Progressive Disclosure로 필요할 때만 열람하여 토큰 과부하 0 달성        │
├─────────────────────────────────────────────────────────────────────────────┤
│ 3. 외부 서드파티 / 표준 연동   ➔ MCP (Model Context Protocol)               │
│    • DB 연결, Wikipedia, GitHub, 기업 사내 레거시 시스템                   │
│    • FastMCP 기반 Stdio / SSE로 프로세스 격리 및 안전한 실행 보장            │
└─────────────────────────────────────────────────────────────────────────────┘
```

## 🧹 Part 6. Clean-up & Reset (실습 샌드박스 정리)

실습용으로 생성했던 임시 샌드박스 디렉토리(`demo_dir`)와 생성 파일을 안전하게 삭제하여 환경을 초기화합니다.

In [ ]:
# Part 6. 샌드박스 임시 디렉토리 정리
if os.path.exists(demo_dir):
    shutil.rmtree(demo_dir, ignore_errors=True)
    print(f"🧹 실습 임시 샌드박스 디렉토리 완전 삭제 완료: {demo_dir}")
else:
    print("ℹ️ 이미 정리되었거나 존재하지 않는 디렉토리입니다.")

print("🎉 Skills & MCP 하네스 실습이 성공적으로 완료되었습니다!")